# local-wlm cloud GPU runner

Run this on **Kaggle** (Settings → Accelerator → GPU T4 ×2) or **Google Colab** (Runtime → Change runtime type → T4 GPU).

It installs Genesis, clones your repo, runs a simulation headlessly on the free GPU, and shows the resulting MP4.

In [ ]:
!nvidia-smi
%pip install -q genesis-world

In [ ]:
# point this at your GitHub repo once you've pushed local-wlm
REPO_URL = "https://github.com/YOUR_USERNAME/local-wlm.git"
!rm -rf local-wlm && git clone {REPO_URL}
%cd local-wlm

In [ ]:
# headless EGL rendering (no display on cloud machines)
import os
os.environ["PYOPENGL_PLATFORM"] = "egl"

!python sims/dominoes.py --gpu --res 1920 1080

In [ ]:
!python sims/water_splash.py --gpu --particle-size 0.008 --res 1920 1080

In [ ]:
# swimmer with mesh-based humanoid (Unitree G1), dense water
!python sims/swimmer_g1.py --gpu --particle-size 0.015 --res 1920 1080

## Beauty render via Blender Cycles (Phase 2)

Export the sim (water particles + robot link poses), rebuild the scene in Blender with a smooth liquid surface and proper materials, and render with Cycles on the same free GPU.

In [ ]:
# 1. export water particles + robot link poses
!python sims/swimmer_g1.py --gpu --particle-size 0.02 --seconds 5 --export export/swim

# 2. get Blender (headless Linux build)
!wget -q https://download.blender.org/release/Blender4.2/blender-4.2.0-linux-x64.tar.xz
!tar xf blender-4.2.0-linux-x64.tar.xz

# 3. import + render with Cycles on the GPU
!./blender-4.2.0-linux-x64/blender -b -P blender/import_swim_g1.py -- \
    --export-dir export/swim --out out/blender/ --render --samples 64

# 4. encode frames to video
!ffmpeg -y -framerate 60 -i out/blender/frame_%04d.png -c:v libx264 -pix_fmt yuv420p out/swimmer_blender.mp4

In [ ]:
# preview inline
from IPython.display import Video
Video("out/dominoes.mp4", embed=True, width=640)

**Downloading results:** on Kaggle, copy videos to `/kaggle/working/` (they appear in the notebook's Output tab). On Colab, use `files.download`.

In [ ]:
!mkdir -p /kaggle/working 2>/dev/null && cp out/*.mp4 /kaggle/working/ 2>/dev/null || true